Data Load

In [9]:
import pandas as pd

# Load the dataset from the URL below
url = "https://raw.githubusercontent.com/DataStorageSchulich/DataStorage/refs/heads/main/dataset_lm.csv"
data = pd.read_csv(url)


Part A: OLS Model

In [10]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS

# Define dependent variable
y = data['Dependent Var']

# Define explanatory variables from #1 to #15
X = data[[f'Explanatory Var #{i}' for i in range(1, 16)]]

# Add a constant term for the intercept
X = sm.add_constant(X)

# Run OLS model
ols_model = OLS(y, X).fit()

# Show summary
ols_summary = ols_model.summary()
print(ols_summary)

                            OLS Regression Results                            
Dep. Variable:          Dependent Var   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.125e+30
Date:                Sat, 12 Oct 2024   Prob (F-statistic):               0.00
Time:                        22:14:32   Log-Likelihood:                 11936.
No. Observations:                 422   AIC:                        -2.384e+04
Df Residuals:                     406   BIC:                        -2.378e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  32.0000   1

Part B: GLS Model Using Error Terms

In [11]:
import numpy as np
from statsmodels.regression.linear_model import OLS, GLS
from statsmodels.tools.tools import add_constant

# Assuming y (dependent variable) and X (independent variables) are defined
# Add constant to independent variables
X = add_constant(X)

# Fit an OLS model
ols_model = OLS(y, X).fit()

# Get residuals from the OLS model
errors = ols_model.resid

# Calculate standard deviation and autocorrelations
std_dev_errors = np.std(errors)
autocorr_lag1 = np.corrcoef(errors[:-1], errors[1:])[0, 1]
autocorr_lag2 = np.corrcoef(errors[:-2], errors[2:])[0, 1]
autocorr_lag3 = np.corrcoef(errors[:-3], errors[3:])[0, 1]

print(f'Standard Deviation of Errors: {std_dev_errors}')
print(f'Autocorrelation (lag 1): {autocorr_lag1}')
print(f'Autocorrelation (lag 2): {autocorr_lag2}')
print(f'Autocorrelation (lag 3): {autocorr_lag3}')

# Now fit a GLS model (assuming autocorrelation exists)
gls_model = GLS(y, X).fit()

# Show GLS summary
gls_summary = gls_model.summary()
print(gls_summary)

Standard Deviation of Errors: 5.555415584649497e-14
Autocorrelation (lag 1): 0.14266367724760337
Autocorrelation (lag 2): 0.00506883595593113
Autocorrelation (lag 3): -0.04787405830765893
                            GLS Regression Results                            
Dep. Variable:          Dependent Var   R-squared:                       1.000
Model:                            GLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.125e+30
Date:                Sat, 12 Oct 2024   Prob (F-statistic):               0.00
Time:                        22:14:32   Log-Likelihood:                 11936.
No. Observations:                 422   AIC:                        -2.384e+04
Df Residuals:                     406   BIC:                        -2.378e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                      

Part C: Lasso Model with Training and Test Split

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_percentage_error

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Run Lasso model with alpha=1
lasso_model = Lasso(alpha=1)
lasso_model.fit(X_train, y_train)

# Estimate coefficients
lasso_coefficients = lasso_model.coef_
print(f'Lasso Coefficients: {lasso_coefficients}')

# Calculate mean absolute percentage error (MAPE)
y_pred = lasso_model.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred)
print(f'Mean Absolute Percentage Error (MAPE): {mape}')

# Find approximate value for alpha to minimize MAPE
from sklearn.model_selection import GridSearchCV

alphas = np.logspace(-4, 1, 50)
lasso_grid = GridSearchCV(Lasso(), {'alpha': alphas}, scoring='neg_mean_absolute_percentage_error', cv=5)
lasso_grid.fit(X_train, y_train)

best_alpha = lasso_grid.best_params_['alpha']
print(f'Best alpha value: {best_alpha}')

Lasso Coefficients: [ 0.          1.26972628  1.68394638  2.02626245  2.08756512 -0.91746375
 -0.          0.         -0.          0.         -0.          0.01314162
  0.         -0.          0.         -0.03617731]
Mean Absolute Percentage Error (MAPE): 0.04432190198291582
Best alpha value: 0.0001
